# CUDA to HIP Conversion Lab

## Abstract

In this lab, you will validate the ROCm environment by migrating a simple CUDA application to HIP and executing it on an AMD GPU. You will learn the fundamentals of CUDA-to-HIP translation, understand the conversion process, build HIP applications, and verify correct GPU kernel execution using the ROCm platform.

## Objectives

After completing this lab, you will be able to:

- Validate the HIP development toolchain including hipcc compiler and hipify-perl translation tool required for CUDA-to-HIP application migration on AMD GPUs
- Create and understand a simple CUDA "Hello World" application that demonstrates basic GPU kernel programming concepts
- Convert CUDA source code to HIP using hipify-perl and understand the automatic translation process
- Build converted HIP applications using the HIP compiler and analyze the differences between CUDA and HIP source code
- Execute HIP kernels on AMD GPUs to confirm correct parallel execution and toolchain functionality

## Introduction

<p style="text-align:justify;">The AMD ROCm software platform provides a comprehensive open-source ecosystem for GPU computing on AMD accelerators, enabling developers to build, port, and run high-performance applications using familiar programming models. ROCm supports heterogeneous computing through technologies such as HIP (Heterogeneous-compute Interface for Portability), which allows CUDA applications to be migrated to run efficiently on AMD GPUs with minimal code changes. This portability layer preserves the CUDA programming model while enabling execution on AMD hardware, making it easier for developers to leverage their existing CUDA expertise and codebase investments.</p>

### HIP and ROCm Software Stack Overview

Understanding the HIP programming model and its position within the ROCm software stack is essential for effective CUDA-to-HIP migration. The ROCm ecosystem is designed with a layered architecture that provides both portability and performance.

<p style="text-align:justify;">At the application level, developers write code using the HIP programming model, which closely mirrors CUDA's syntax and semantics. This application code can interact directly with HIP or through higher-level libraries such as hipBLAS, rocBLAS, and other ROCm-optimized libraries that provide common computational primitives. These libraries offer highly optimized implementations of linear algebra, FFT, and other mathematical operations specifically tuned for AMD GPU architectures.</p>

<p style="text-align:justify;">The HIP layer itself acts as a portability interface between application code and the underlying AMD runtime. HIP provides two main components: the runtime API and the kernel language. The runtime API handles device management, memory allocation, kernel launches, and synchronization operations using function calls that are nearly identical to their CUDA equivalents. The kernel language allows developers to write device code using the same parallel programming constructs they are familiar with from CUDA, including thread indexing, shared memory, and synchronization primitives.</p>

<p style="text-align:justify;">Below the HIP abstraction layer sits the AMD runtime, which manages the actual interaction with AMD GPU hardware. This runtime handles low-level details such as command queue management, kernel compilation and dispatch, memory transfers, and hardware resource allocation. The runtime interfaces with the AMD platform layer, which includes the GPU drivers and hardware abstraction interfaces that provide direct access to GPU compute units, memory controllers, and other hardware resources.</p>

<p style="text-align:justify;">This layered design enables developers to preserve their existing CUDA programming model and code structure while transitioning execution to AMD hardware. Application code written in CUDA can be automatically or semi-automatically converted to HIP, with the conversion primarily involving simple API substitutions while maintaining the same execution semantics, parallel decomposition strategies, and algorithmic approaches.</p>

### Lab Overview

This lab introduces you to the ROCm development workflow through a practical, hands-on validation of the CUDA-to-HIP migration process. The lab focuses on environment readiness and fundamental CUDA-to-HIP conversion using a simple "Hello World" CUDA kernel.

<p style="text-align:justify;">You will begin by verifying that the HIP development toolchain is properly installed and accessible. This includes checking the HIP compiler (hipcc) and the CUDA-to-HIP translation tool (hipify-perl), both of which are essential components for migrating and building HIP applications. Verifying these tools ensures that your environment is correctly configured before attempting any conversion or compilation tasks.</p>

<p style="text-align:justify;">Next, you will create a basic CUDA program that demonstrates fundamental GPU programming concepts. This "Hello World" kernel launches multiple GPU threads in parallel, with each thread printing a message that includes its unique thread ID. This simple example illustrates the core structure of a CUDA program, including host-side kernel launch configuration, device-side kernel execution, GPU-CPU synchronization, and device-side output from GPU threads. Understanding this baseline CUDA implementation provides context for evaluating the conversion process.</p>

<p style="text-align:justify;">Once the CUDA program is written and understood, you will use AMD's hipify-perl tool to automatically translate the CUDA source code to HIP. The hipify-perl tool performs text-based substitutions, replacing CUDA-specific API calls and includes with their HIP equivalents. You will first analyze the CUDA source without modification to understand what will be converted, then perform the actual translation. After conversion, you will examine the differences between the original CUDA code and the translated HIP code, gaining insight into how the conversion process works and what changes are required at the source level.</p>

<p style="text-align:justify;">With the HIP source code generated, you will compile it using the HIP compiler (hipcc) to create an executable binary. The hipcc compiler handles both host-side C++ code and device-side kernel code, generating appropriate machine code for the AMD GPU architecture. This compilation step validates that the converted code is syntactically correct and that the HIP toolchain can successfully build applications from HIP source.</p>

<p style="text-align:justify;">Finally, you will execute the compiled HIP binary on an AMD GPU to confirm correct kernel execution and toolchain functionality. When the program runs, the GPU kernel launches with multiple threads executing in parallel, and each thread prints its message to the console. Observing this output confirms that the kernel executed successfully on the GPU, that device-side printf works correctly, and that the entire end-to-end workflow—from CUDA source authoring through HIP conversion, compilation, and execution—functions as expected.</p>

<p style="text-align:justify;">Through this hands-on lab, you will gain practical experience with CUDA-to-HIP migration and ROCm toolchain validation. The lab provides a solid foundation for understanding how existing CUDA applications can be ported to AMD GPUs, what the conversion process involves, and how to build and execute HIP applications on the ROCm platform. These skills are essential for developers transitioning CUDA codebases to AMD hardware or validating ROCm readiness for GPU computing workloads.</p>

## Environment Readiness and CUDA-to-HIP Conversion

In this section, you will use HIP tools to verify the toolchain, convert a minimal CUDA "hello" kernel to HIP, build it with hipcc, and run the binary to confirm end-to-end HIP execution. This workflow demonstrates the complete CUDA-to-HIP migration process from source translation through execution on AMD GPU hardware.

### Verify the HIP Development Toolchain

Before attempting any CUDA-to-HIP conversion or compilation, you must first verify that the necessary HIP development tools are properly installed and accessible in your environment. The HIP toolchain includes the HIP compiler (hipcc) for building applications and the hipify-perl tool for translating CUDA source code to HIP.

Display the HIP compiler version to confirm that hipcc is available and properly configured in your system PATH.

In [ ]:
!hipcc --version

This command prints the installed HIP compiler's version and build details, confirming that the HIP toolchain is available before attempting any HIP builds. You should see version information that includes details about the HIP/Clang compiler infrastructure. If the command is not found, you will need to fix your PATH environment variable or install HIP before proceeding with the lab.

Verify that the hipify-perl translation tool is also available. This tool is essential for automatically converting CUDA source code to HIP.

In [ ]:
!which hipify-perl

This command shows the filesystem path to hipify-perl, ensuring it is available before converting any CUDA code. You should see a path displayed indicating where the tool is installed. If the output is empty, you will need to install or properly configure the HIP utilities before proceeding.

### Create a Working Directory

Organize your work by creating a dedicated directory structure for this lab. A clean workspace helps keep artifacts tidy and makes the workflow more reproducible.

In [ ]:
!mkdir -p cuda_to_hip_lab/cuda_hello
%cd cuda_to_hip_lab/cuda_hello

The `mkdir -p` command creates the directory structure, with the `-p` flag ensuring that parent directories are also created if they don't exist. The `%cd` magic command changes the current working directory so all remaining file operations and builds occur inside this workspace. Your Jupyter prompt should now reflect this working directory.

### Create the CUDA Source File

You will now create a simple CUDA "Hello World" program that demonstrates the basic structure of a CUDA application. This program will launch a GPU kernel with multiple threads, where each thread prints a message including its unique thread ID.

In [ ]:
%%writefile hello_cuda.cu
#include <cuda_runtime.h>
#include <iostream>
#include <cstdio>

__global__ void helloKernel()
{
    int tid = threadIdx.x;
    printf("Hello from CUDA GPU kernel! Thread ID = %d\n", tid);
}

int main()
{
    std::cout << "Launching CUDA kernel with 1 block and 64 threads..." << std::endl;
    
    // Launch kernel with 1 block, 64 threads
    helloKernel<<<1, 64>>>();
    
    // Wait for GPU to finish
    cudaDeviceSynchronize();
    
    std::cout << "Kernel execution finished." << std::endl;
    return 0;
}

### Understanding the CUDA Source Code

<p style="text-align:justify;">This program demonstrates a simple CUDA "Hello World" application that illustrates the basic structure of a CUDA program, including host code execution, GPU kernel definition, kernel launch configuration, and synchronization between the host and device.</p>

<p style="text-align:justify;">The program begins by including the necessary header files to enable both CUDA runtime functionality and standard input/output operations. The `cuda_runtime.h` header provides access to the CUDA runtime programming model, including kernel launch support using the `<<<>>>` syntax, device synchronization through functions such as `cudaDeviceSynchronize`, and other runtime-managed features. The `iostream` header enables C++ stream-based output using `std::cout` for host-side messages, while the `cstdio` header provides C-style I/O support through `printf`, which is required because the GPU kernel prints output directly from the device. Device-side `printf` is supported on modern GPUs with Compute Capability 2.0 or higher.</p>

<p style="text-align:justify;">The code then defines a CUDA kernel using the `__global__` qualifier, which indicates that the function executes on the GPU and is launched from the host. This kernel runs in parallel across multiple GPU threads. Inside the kernel, the built-in CUDA variable `threadIdx.x` is used to identify each thread's index within its block along the x-dimension. Each thread stores this value in a local variable and prints a message that includes its unique thread ID. Because many threads execute concurrently on the GPU, the order in which these messages appear in the output is not deterministic, reflecting the parallel nature of GPU execution.</p>

<p style="text-align:justify;">Execution begins in the `main()` function, which serves as the entry point for the host (CPU) program. All CUDA runtime calls and kernel launches are initiated from this function. Before launching the kernel, the program prints a status message using `std::cout` to inform the user of the grid and block configuration that will be used. This makes the execution flow easier to follow and helps distinguish host-side output from device-side output when the program runs.</p>

<p style="text-align:justify;">The kernel is launched using the CUDA runtime execution configuration syntax `<<<gridDim, blockDim>>>`, where the first parameter specifies the number of blocks in the grid and the second parameter specifies the number of threads per block. In this case, the kernel is launched with one block containing 64 threads, resulting in a total of 64 GPU threads executing the kernel in parallel. Kernel launches in CUDA are asynchronous by default, meaning the host queues the kernel for execution on the GPU and continues running without waiting for the kernel to complete unless explicit synchronization is requested.</p>

<p style="text-align:justify;">To ensure correct program behavior, the host calls `cudaDeviceSynchronize()` after launching the kernel. This call blocks the CPU until all previously issued GPU work has completed, including the kernel execution. Synchronization ensures that all device-side `printf` output is flushed to the host console before the program proceeds and that any deferred kernel launch or execution errors are reported back to the host at this point.</p>

<p style="text-align:justify;">Finally, after synchronization confirms that the GPU work has completed, the program prints a completion message from the host to indicate successful kernel execution. The `return 0;` statement then terminates the application and returns a success status to the host operating system, signaling that the program executed without errors.</p>

### Analyze the CUDA Source Without Modification

Before performing the conversion, it is useful to analyze the CUDA source file to understand what elements will be converted and to estimate the migration effort. The hipify-perl tool provides an "examine" mode that performs this analysis without modifying the source file.

In [ ]:
!hipify-perl --examine hello_cuda.cu

This command analyzes the CUDA source file without modifying it and reports convertible CUDA APIs, total lines of code (LOC), and any warnings or issues that might affect conversion. The examine mode is useful for estimating migration effort before performing the actual conversion, as it identifies which CUDA-specific elements will need to be replaced with HIP equivalents.

### Convert CUDA to HIP Using hipify-perl

Now perform the actual CUDA-to-HIP conversion using the hipify-perl tool. This tool performs text-based substitutions, replacing CUDA-specific runtime headers, API calls, and language constructs with their HIP equivalents.

In [ ]:
!hipify-perl hello_cuda.cu > hello_hip.cpp

<p style="text-align:justify;">This command converts CUDA APIs and syntax to their HIP equivalents. The output redirection operator (`>`) writes the converted HIP source to a new file named `hello_hip.cpp`. An alternative approach would be to use the `--inplace` flag, which updates the original file directly and saves a backup of the CUDA source with a `.prehip` extension. No compilation occurs at this stage—this is purely a source code translation step.</p>

<p style="text-align:justify;">The hipify-perl tool performs straightforward text substitutions, replacing patterns like `cuda_runtime.h` with `hip/hip_runtime.h` and `cudaDeviceSynchronize` with `hipDeviceSynchronize`. This automatic translation handles the vast majority of CUDA-to-HIP conversions for runtime API calls, making the migration process efficient and reducing manual effort.</p>

### Inspect the Converted HIP Source

Examine the translated HIP source code to understand what changes were made during the conversion process.

In [ ]:
!cat hello_hip.cpp

This displays the HIP-converted source code, allowing you to see the transformed headers, API calls, and kernel code. The conversion primarily affects runtime includes and synchronization calls, while kernel logic and launch syntax typically remain unchanged.

### Compare CUDA and HIP Source Code

To better understand the conversion process, view a side-by-side comparison of the original CUDA source and the converted HIP source using a unified diff format.

In [ ]:
!diff -u hello_cuda.cu hello_hip.cpp

This command shows a unified diff of the CUDA versus HIP source, making the translation changes explicit and providing valuable insight into the conversion process. The output uses lines prefixed with `-` to indicate removed CUDA code and lines prefixed with `+` to indicate added HIP code.

### Understanding the CUDA-to-HIP Conversion

<p style="text-align:justify;">The diff output reveals several key changes introduced when converting the original CUDA source file to HIP using hipify-perl. CUDA-specific runtime headers such as `cuda_runtime.h` are replaced with HIP runtime headers (`hip/hip_runtime.h`), which provide equivalent functionality for kernel launches, device synchronization, and other runtime features on AMD GPUs. An additional standard header may be included to ensure compatibility during compilation.</p>

<p style="text-align:justify;">The kernel launch syntax remains unchanged, demonstrating that HIP preserves CUDA's execution model and programming structure. The familiar `<<<gridDim, blockDim>>>` syntax continues to work identically in HIP, requiring no modifications to kernel launch configurations. This syntax preservation is one of HIP's key design goals, allowing developers to maintain their existing mental models and code patterns when transitioning from CUDA.</p>

<p style="text-align:justify;">CUDA runtime API calls such as `cudaDeviceSynchronize()` are translated directly to their HIP equivalents (`hipDeviceSynchronize()`), maintaining the same synchronization behavior while targeting the HIP runtime instead of the CUDA runtime. All host-side C++ logic, including output statements and program control flow, remains unchanged. The kernel code itself, including thread indexing and device-side `printf` calls, also requires no modification.</p>

<p style="text-align:justify;">Overall, this conversion illustrates that CUDA-to-HIP migration primarily involves substituting runtime APIs while preserving kernel code, launch configuration, and the overall parallel programming model. For simple applications like this "Hello World" example, the conversion is entirely automatic and requires no manual intervention or code restructuring.</p>

### Build the HIP Executable

Compile the converted HIP source code using the HIP compiler to create an executable binary. The hipcc compiler handles both host-side C++ code and device-side kernel code, generating appropriate machine code for AMD GPU architecture.

In [ ]:
!hipcc hello_hip.cpp -o hello_hip

<p style="text-align:justify;">This command compiles the HIP source into an executable named `hello_hip`, validating that the HIP toolchain and the converted code are working correctly. You may see some compiler warnings during the build process, which can typically be ignored for this simple example. Upon successful compilation, an executable file named `hello_hip` will be created in the current directory.</p>

The hipcc compiler is a wrapper that invokes the appropriate backend compiler for both host and device code. For AMD GPUs, hipcc uses Clang-based compilation to generate GPU code that targets the specific AMD GPU architecture in your system.

Verify that the executable was created successfully by listing the files in the current directory.

In [ ]:
!ls -lh

<p style="text-align:justify;">This command lists files in the working directory with human-readable file sizes, allowing you to verify that the `hello_hip` executable (binary) and source files exist before execution. You should see `hello_cuda.cu` (original CUDA source), `hello_hip.cpp` (converted HIP source), and `hello_hip` (compiled executable) in the listing.</p>

### Execute the HIP Application

Run the compiled HIP executable to confirm that the kernel executes correctly on the AMD GPU. This final step validates the complete end-to-end workflow from CUDA source through HIP conversion, compilation, and execution.

In [ ]:
!./hello_hip

This command runs the HIP binary, which launches a GPU kernel that prints a message from each of the 64 threads executing in parallel. Successful execution completes the end-to-end validation workflow: author CUDA source → convert to HIP using hipify → build with hipcc → execute on AMD GPU.

<p style="text-align:justify;">The output should begin with a host-side message indicating that the kernel is being launched with 1 block and 64 threads. This is followed by approximately 64 per-thread messages from the GPU kernel, each displaying "Hello from CUDA GPU kernel! Thread ID = X" where X ranges from 0 to 63. The order of these messages may vary because GPU threads execute in parallel and the order in which their output appears is non-deterministic. Finally, a host-side completion message indicates that the kernel has finished executing and the program has completed successfully.</p>

<p style="text-align:justify;">This output confirms several important aspects of the system: the HIP runtime is correctly configured, the AMD GPU is accessible and functioning, device-side `printf` from GPU kernels works correctly, kernel launch and synchronization mechanisms are operating as expected, and the complete CUDA-to-HIP migration workflow is functional from source translation through execution.</p>

### Return to Home Directory

Clean up your working environment by returning to the home directory.

In [ ]:
%cd ../..

This navigates two directory levels up, returning you to your original working directory and completing the lab workflow.

## Summary

<p style="text-align:justify;">In this lab, you validated the HIP development toolchain and performed a complete CUDA-to-HIP migration workflow. You verified that the hipcc compiler and hipify-perl translation tool were properly installed and accessible in your environment, ensuring that all necessary components were available before beginning the migration process.</p>

<p style="text-align:justify;">You created a simple CUDA "Hello World" application that demonstrated fundamental GPU programming concepts, including parallel kernel execution, thread indexing, device-side output, and host-device synchronization. This baseline CUDA implementation provided the foundation for understanding the conversion process.</p>

<p style="text-align:justify;">Using AMD's hipify-perl tool, you automatically converted the CUDA source code to HIP, replacing CUDA-specific runtime headers and API calls with their HIP equivalents. You examined the differences between the original CUDA code and the converted HIP code, gaining insight into how the translation process works and what changes occur at the source level. The comparison revealed that HIP preserves the CUDA programming model, including kernel launch syntax and kernel code, while substituting runtime API calls.</p>

<p style="text-align:justify;">You successfully compiled the converted HIP source code using the hipcc compiler, generating an executable binary that targets AMD GPU hardware. Finally, you executed the HIP application on an AMD GPU and observed the parallel execution of 64 GPU threads, each printing its unique message. This confirmed that the kernel executed correctly and that the entire end-to-end workflow—from CUDA source authoring through HIP conversion, compilation, and execution—functioned as expected.</p>

<p style="text-align:justify;">Through this hands-on lab, you gained practical experience with the fundamentals of CUDA-to-HIP migration and ROCm toolchain validation. You learned that HIP provides a high degree of source-level compatibility with CUDA, that the automatic conversion process handles most common CUDA constructs, and that migrated applications can execute on AMD GPUs with minimal code changes. These skills provide a solid foundation for transitioning existing CUDA applications to AMD GPUs and for developing portable GPU applications that can target multiple vendor platforms using the HIP abstraction layer.</p>